In [29]:
# Copyright 2024 NVIDIA Corporation. All Rights Reserved.
# Adapted for PMC-Patients Clinical Triplet Distillation on Apple Silicon (M4 / MLX) and Scalable Multi-GPU Clusters.

# Each user is responsible for checking the content of datasets and the
# applicable licenses and determining if suitable for the intended use.

<img src="http://developer.download.nvidia.com/compute/machine-learning/frameworks/nvidia_logo.png" style="width:60px; float:right"><br>
# <font color="#76b900">**Finetuning LLM for Clinical Triplet Prediction**<br/>Apple Silicon (M4 / MLX) Local Run & Scaled Cluster (NeMo / NIM)</font>

This notebook fine-tunes **`Meta-Llama-3-8B-Instruct`** for automated **clinical entity-relation triplet extraction** using patient case narratives from the **[PMC-Patients Dataset](https://huggingface.co/datasets/zhengyun21/PMC-Patients)** (`eval/data/pmc_cut.jsonl`).

#### **Execution Modes**
1. **Local Apple Silicon (M4 MacBook Pro with 24GB Unified Memory)**: Fast, memory-efficient 4-bit LoRA fine-tuning using Apple's native **MLX** framework (`mlx-lm`) on Metal Performance Shaders (MPS). Peak training memory: **~8–10 GB**, leaving >14 GB headroom for macOS.
2. **Cloud / Enterprise Multi-GPU Cluster**: Scaled-out training with **NVIDIA NeMo (Megatron-LM)** and serving via **NVIDIA Inference Microservices (NIM)**.

### Learning Objectives 
1. **Clinical Distillation Dataset Preparation**: Ingest `eval/data/pmc_cut.jsonl` clinical patient records and wrap into the Llama-3 Chat Template format (`<|begin_of_text|><|start_header_id|>user...`).
2. **Apple Silicon Hardware Optimization**: LoRA fine-tune `Meta-Llama-3-8B-Instruct-4bit` locally using Apple's Metal Performance Shaders (MPS) via `mlx-lm` within a 24GB memory footprint.
3. **Local Testing & Validation**: Test and evaluate the fine-tuned LoRA adapter (`adapters.npz`) directly on clinical patient test cases on Mac without Docker overhead.
4. **Cloud / Cluster Scale-Up Pathway**: Transition from local MLX prototyping to multi-GPU (8x H100) NeMo Megatron training and NVIDIA NIM microservice deployment.

### **Why Llama-3-8B on a 24GB Unified Memory Footprint?**
For a 24GB Unified Memory MacBook Pro (Apple Silicon M4), `Meta-Llama-3-8B-Instruct` is the ideal distillation target:
- **Baseline 8B in 16-bit (fp16/bf16)** requires ~16GB of VRAM just for base weights, which would cause swapping/OOM during training backward passes.
- **MLX 4-bit quantized base model (`Meta-Llama-3-8B-Instruct-4bit`)** requires only **~4.5GB VRAM**.
- **LoRA Adapter (rank 16, 16 layers)**: only ~50MB of trainable parameters.
- **Gradients + AdamW optimizer states**: ~1.5 - 2.0GB.
- **Context tokens & activations (batch size 4)**: ~2 - 3GB.
- **Total training peak memory**: **~8 - 10GB**, leaving >14GB of headroom for macOS with zero swap overhead.

#### **MLX vs NVIDIA NeMo: Architecture Comparison**
| Feature | Local Apple Silicon (MacBook M4) | Scale-Out Enterprise (DGX / Cloud) |
| :--- | :--- | :--- |
| **Engine** | Apple **MLX** (`mlx-lm`) | **NVIDIA NeMo** (Megatron-LM) |
| **Hardware** | Apple M4 GPU (Unified Memory / Metal) | NVIDIA GPUs (A100 / H100 SXM) |
| **Dataset** | Clinical cases from `eval/data/pmc_cut.jsonl` | Distributed Sharded JSONL |
| **Quantization** | 4-bit native Metal dequantization | BF16 / FP8 Megatron O2 |
| **Parallelism** | Single-node Unified Memory zero-copy | Tensor Parallel (TP) + Pipeline (PP) |
| **Inference** | `mlx-lm.generate` / local Python API | **NVIDIA NIM** (vLLM / TensorRT-LLM) |

<br/>
## <font color="#76b900">**0. Environment Setup & Hardware Detection**</font>

In [30]:
import os
import sys
import json
import re
import random
from pprint import pprint
import subprocess
import urllib.request
import requests

def detect_runtime():
    print(f"Python: {sys.version.split()[0]} ({sys.platform})")
    
    # Check Apple Silicon / MLX
    is_mac = sys.platform == "darwin"
    has_mlx = False
    try:
        import mlx.core as mx
        print(f"✅ Apple Silicon MLX available (v{mx.__version__}) - Metal GPU acceleration active")
        has_mlx = True
    except ImportError:
        if is_mac:
            print("ℹ️ Apple Silicon detected, but 'mlx' / 'mlx-lm' not installed yet.")
            print("   Run the cell below to install MLX.")
            
    # Check NVIDIA CUDA
    has_cuda = False
    try:
        import torch
        if torch.cuda.is_available():
            print(f"✅ NVIDIA CUDA available: {torch.cuda.get_device_name(0)}")
            has_cuda = True
    except ImportError:
        pass
        
    return {"is_mac": is_mac, "has_mlx": has_mlx, "has_cuda": has_cuda}

runtime_info = detect_runtime()

Python: 3.11.15 (darwin)
✅ Apple Silicon MLX available (v0.32.2) - Metal GPU acceleration active


In [31]:
# Install Apple MLX framework for local Mac training (run once if needed)
# !pip install -q mlx-lm mlx

**Path & Clinical Data Source Configuration**:
We configure paths dynamically to locate the clinical evaluation dataset generated by the `data_prep` pipeline: **`eval/data/pmc_cut.jsonl`**.

In [32]:
import os

NOTEBOOK_DIR = os.getcwd()
BASE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
WORKSPACE_ROOT = os.path.abspath(os.path.join(BASE_DIR, ".."))

# Locate PMC-Patients input cut (from backend/eval/data or local GTC25_DLI/data)
BACKEND_PMC_CUT = os.path.join(WORKSPACE_ROOT, "backend", "eval", "data", "pmc_cut.jsonl")
LOCAL_PMC_CUT = os.path.join(BASE_DIR, "data", "pmc_cut.jsonl")

if os.path.exists(BACKEND_PMC_CUT):
    PMC_SOURCE_JSONL = BACKEND_PMC_CUT
elif os.path.exists(LOCAL_PMC_CUT):
    PMC_SOURCE_JSONL = LOCAL_PMC_CUT
else:
    PMC_SOURCE_JSONL = BACKEND_PMC_CUT

# Training dataset directories (for fine-tuning)
DATA_DIR = os.getenv("DATA_DIR", os.path.join(BASE_DIR, "data"))
TRAINING_DATA_DIR = os.path.join(DATA_DIR, "training_data")
os.makedirs(TRAINING_DATA_DIR, exist_ok=True)
OUTPUT_JSONL = os.path.join(TRAINING_DATA_DIR, "output.jsonl")

# Model & LoRA storage directories
MODEL_DIR = os.path.join(BASE_DIR, "model")
LORA_DIR_MLX = os.path.join(MODEL_DIR, "loras", "Meta-Llama-3-8B-Instruct-PMC-LoRA-mlx")
os.makedirs(LORA_DIR_MLX, exist_ok=True)

print(f"PMC Input Dataset:       {PMC_SOURCE_JSONL} (exists: {os.path.exists(PMC_SOURCE_JSONL)})")
print(f"Fine-Tuning Dataset Dir: {TRAINING_DATA_DIR}")
print(f"MLX Adapter Output Dir:  {LORA_DIR_MLX}")

PMC Input Dataset:       /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/backend/eval/data/pmc_cut.jsonl (exists: True)
Fine-Tuning Dataset Dir: /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/GTC25_DLI/data/training_data
MLX Adapter Output Dir:  /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/GTC25_DLI/model/loras/Meta-Llama-3-8B-Instruct-PMC-LoRA-mlx


<br/>
## <font color="#76b900">**1. Clinical Dataset Distillation & Chat Template Formatting**</font>

import os
import json

# Comprehensive Clinical Mapping Rules for Triplets Extraction
SYMPTOM_MAP = {
    "fever": "Fever",
    "cough": "Cough",
    "dyspnea": "Dyspnea",
    "shortness of breath": "Dyspnea",
    "headache": "Headache",
    "fatigue": "Fatigue",
    "chest pain": "Chest Pain",
    "abdominal pain": "Abdominal Pain",
    "nausea": "Nausea",
    "vomiting": "Vomiting",
    "diarrhea": "Diarrhea",
    "rash": "Rash",
    "hearing loss": "Hearing Loss",
    "loss of appetite": "Anorexia"
}

TREATMENT_MAP = {
    "oxygen": "Oxygen Therapy",
    "antibiotic": "Antibiotic Therapy",
    "surgery": "Surgical Intervention",
    "ventilat": "Mechanical Ventilation",
    "rehabilitation": "Rehabilitation",
    "appendectomy": "Appendectomy",
    "paracetamol": "Analgesic",
    "physical therapy": "Physical Therapy"
}

def extract_clinical_triplets_from_case(patient):
    """Extract standardized clinical triplets from patient record."""
    pid = patient.get("patient_id", "0")
    subject = f"Patient {pid}"
    text = patient.get("text", "")
    age = patient.get("age", "")
    gender = patient.get("gender", "")
    cond = patient.get("condition_hint", "")
    
    triplets = []
    
    # 1. Demographics
    if age:
        triplets.append((subject, "PERSON", "Has_Age", age, "METRIC"))
    if gender:
        triplets.append((subject, "PERSON", "Has_Gender", gender.capitalize(), "CONCEPT"))
        
    # 2. Diagnosed Condition
    if cond:
        triplets.append((subject, "PERSON", "Diagnosed_With", cond.upper(), "CONDITION"))
    elif "covid-19" in text.lower():
        triplets.append((subject, "PERSON", "Diagnosed_With", "COVID-19", "CONDITION"))
    elif "ards" in text.lower():
        triplets.append((subject, "PERSON", "Diagnosed_With", "ARDS", "CONDITION"))
        
    # 3. Presenting Symptoms
    text_lower = text.lower()
    for kw, label in SYMPTOM_MAP.items():
        if kw in text_lower:
            triplet = (subject, "PERSON", "Presents_With", label, "CONDITION")
            if triplet not in triplets:
                triplets.append(triplet)
                
    # 4. Treatments & Interventions
    for kw, label in TREATMENT_MAP.items():
        if kw in text_lower:
            triplet = (subject, "PERSON", "Receives", label, "TREATMENT")
            if triplet not in triplets:
                triplets.append(triplet)
                
    return triplets

def process_clinical_dataset_for_finetuning(input_jsonl, output_jsonl):
    """Convert clinical patients into formatted Llama-3 instruction fine-tuning entries."""
    if not os.path.exists(input_jsonl):
        print(f"Notice: {input_jsonl} not found. Run 2_SEC_Data_Preparation.ipynb first.")
        sample_patient = {
            "patient_id": "0", "age": "60 years", "gender": "male", "condition_hint": "ards",
            "text": "This 60-year-old male was hospitalized due to moderate ARDS from COVID-19 with symptoms of fever, dry cough, and dyspnea. Physical therapy and oxygen therapy were administered."
        }
        records = [sample_patient]
    else:
        with open(input_jsonl, 'r', encoding='utf-8') as f:
            records = [json.loads(line) for line in f if line.strip()]
            
    print(f"Loaded {len(records)} clinical patient cases from {input_jsonl}")
    
    os.makedirs(os.path.dirname(output_jsonl), exist_ok=True)
    count = 0
    with open(output_jsonl, 'w', encoding='utf-8') as out_f:
        for r in records:
            raw_text = r.get("text", "").strip()
            if not raw_text:
                continue
                
            triplets = extract_clinical_triplets_from_case(r)
            response_str = str(triplets)
            
            prompt_instruction = f"Extract all clinical entity-relation-entity triplets from the following patient case as a valid Python list:\n\n{raw_text}"
            
            # Standard Llama-3 instruction chat template components
            prompt_formatted = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{prompt_instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
            completion_formatted = f"{response_str}<|eot_id|>"
            
            # JSONL entry supporting both MLX CompletionsDataset (prompt+completion) and raw text/input/output
            jsonl_entry = {
                "prompt": prompt_formatted,
                "completion": completion_formatted,
                "text": f"{prompt_formatted}{completion_formatted}",
                "input": raw_text,
                "output": response_str,
                "patient_id": r.get("patient_id", str(count))
            }
            out_f.write(json.dumps(jsonl_entry, ensure_ascii=False) + '\n')
            count += 1
            
    print(f"✅ Generated {count} fine-tuning records with prompt/completion fields -> {output_jsonl}")

# Run clinical dataset distillation formatting
process_clinical_dataset_for_finetuning(PMC_SOURCE_JSONL, OUTPUT_JSONL)


In [33]:
import os
import json
import re
from pprint import pprint

# Clinical dictionary for rule-grounded triplet bootstrapping
SYMPTOM_MAP = {
    "fever": "Fever", "cough": "Cough", "dry cough": "Dry Cough",
    "dyspnea": "Dyspnea", "breathless": "Dyspnea", "breathlessness": "Dyspnea",
    "pain": "Pain", "chest pain": "Chest Pain", "headache": "Headache",
    "fatigue": "Fatigue", "weakness": "Weakness", "anxiety": "Anxiety"
}

TREATMENT_MAP = {
    "physical therapy": "Physical Therapy", "breathing exercise": "Breathing Exercises",
    "breathing exercises": "Breathing Exercises", "prone position": "Prone Positioning",
    "oxygen": "Oxygen Therapy", "ventilation": "Mechanical Ventilation",
    "intubation": "Intubation", "antibiotics": "Antibiotic Therapy",
    "surgery": "Surgical Intervention", "rehabilitation": "Rehabilitation"
}

def extract_clinical_triplets_from_case(patient):
    """Extract standardized clinical triplets from patient record."""
    pid = patient.get("patient_id", "0")
    subject = f"Patient {pid}"
    text = patient.get("text", "")
    age = patient.get("age", "")
    gender = patient.get("gender", "")
    cond = patient.get("condition_hint", "")
    
    triplets = []
    
    # 1. Demographics
    if age:
        triplets.append((subject, "PERSON", "Has_Age", age, "METRIC"))
    if gender:
        triplets.append((subject, "PERSON", "Has_Gender", gender.capitalize(), "CONCEPT"))
        
    # 2. Diagnosed Condition
    if cond:
        triplets.append((subject, "PERSON", "Diagnosed_With", cond.upper(), "CONDITION"))
    elif "covid-19" in text.lower():
        triplets.append((subject, "PERSON", "Diagnosed_With", "COVID-19", "CONDITION"))
    elif "ards" in text.lower():
        triplets.append((subject, "PERSON", "Diagnosed_With", "ARDS", "CONDITION"))
        
    # 3. Presenting Symptoms
    text_lower = text.lower()
    for kw, label in SYMPTOM_MAP.items():
        if kw in text_lower:
            triplet = (subject, "PERSON", "Presents_With", label, "CONDITION")
            if triplet not in triplets:
                triplets.append(triplet)
                
    # 4. Treatments & Interventions
    for kw, label in TREATMENT_MAP.items():
        if kw in text_lower:
            triplet = (subject, "PERSON", "Receives", label, "TREATMENT")
            if triplet not in triplets:
                triplets.append(triplet)
                
    return triplets

def process_clinical_dataset_for_finetuning(input_jsonl, output_jsonl):
    """Convert clinical patients into formatted Llama-3 instruction fine-tuning entries."""
    if not os.path.exists(input_jsonl):
        print(f"Notice: {input_jsonl} not found. Run 2_SEC_Data_Preparation.ipynb first.")
        # Fallback to high-quality clinical seed sample
        sample_patient = {
            "patient_id": "0", "age": "60 years", "gender": "male", "condition_hint": "ards",
            "text": "This 60-year-old male was hospitalized due to moderate ARDS from COVID-19 with symptoms of fever, dry cough, and dyspnea. Physical therapy and oxygen therapy were administered."
        }
        records = [sample_patient]
    else:
        with open(input_jsonl, 'r', encoding='utf-8') as f:
            records = [json.loads(line) for line in f if line.strip()]
            
    print(f"Loaded {len(records)} clinical patient cases from {input_jsonl}")
    
    os.makedirs(os.path.dirname(output_jsonl), exist_ok=True)
    count = 0
    with open(output_jsonl, 'w', encoding='utf-8') as out_f:
        for r in records:
            raw_text = r.get("text", "").strip()
            if not raw_text:
                continue
                
            triplets = extract_clinical_triplets_from_case(r)
            response_str = str(triplets)
            
            prompt = f"Extract all clinical entity-relation-entity triplets from the following patient case as a valid Python list:\n\n{raw_text}"
            
            # Standard Llama-3 instruction chat template
            jsonl_entry = {
                "text": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{response_str}<|eot_id|>",
                "input": raw_text,
                "output": response_str,
                "patient_id": r.get("patient_id", str(count))
            }
            out_f.write(json.dumps(jsonl_entry, ensure_ascii=False) + '\n')
            count += 1
            
    print(f"✅ Generated {count} fine-tuning records -> {output_jsonl}")

# Run clinical dataset distillation formatting
process_clinical_dataset_for_finetuning(PMC_SOURCE_JSONL, OUTPUT_JSONL)

Loaded 100 clinical patient cases from /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/backend/eval/data/pmc_cut.jsonl
✅ Generated 100 fine-tuning records -> /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/GTC25_DLI/data/training_data/output.jsonl


### Split Clinical Dataset into Train, Validation, and Test
Apple MLX (`mlx-lm`) expects data splits named **`train.jsonl`**, **`valid.jsonl`**, and **`test.jsonl`** inside the training data directory.

In [34]:
import os

# MLX standard files
TRAIN_FILE = os.path.join(TRAINING_DATA_DIR, "train.jsonl")
VALID_FILE = os.path.join(TRAINING_DATA_DIR, "valid.jsonl")
TEST_FILE = os.path.join(TRAINING_DATA_DIR, "test.jsonl")

# Legacy/NeMo alias files
PMC_TRAIN_FILE = os.path.join(TRAINING_DATA_DIR, "pmc_train.jsonl")
PMC_VALID_FILE = os.path.join(TRAINING_DATA_DIR, "pmc_val.jsonl")
PMC_TEST_FILE = os.path.join(TRAINING_DATA_DIR, "pmc_test.jsonl")

In [35]:
import os
import random

def split_jsonl(input_file, train_file, valid_file, test_file, sec_train_file=None, sec_valid_file=None, sec_test_file=None, train_ratio=0.8, valid_ratio=0.1, test_ratio=0.1):
    """Splits JSONL into train, valid, and test sets and creates legacy copies."""
    import random
    if not os.path.exists(input_file):
        print(f"Input file not found: {input_file}")
        return

    with open(input_file, 'r', encoding='utf-8') as f:
        lines = [line for line in f if line.strip()]

    if not lines:
        print("Input file is empty.")
        return

    random.seed(42)
    random.shuffle(lines)

    total_lines = len(lines)
    train_split = max(1, int(total_lines * train_ratio))
    valid_split = max(1, int(total_lines * valid_ratio)) if total_lines > 2 else 0

    train_data = lines[:train_split]
    valid_data = lines[train_split:train_split + valid_split] if valid_split else lines[:1]
    test_data = lines[train_split + valid_split:] if (train_split + valid_split) < total_lines else lines[:1]

    for path, data, name in [(train_file, train_data, "Train"), (valid_file, valid_data, "Validation"), (test_file, test_data, "Test")]:
        with open(path, 'w', encoding='utf-8') as out_f:
            out_f.writelines(data)
        print(f"{name} set: {len(data)} records -> {path}")

    # Duplicate to alias filenames
    if sec_train_file:
        with open(sec_train_file, 'w', encoding='utf-8') as f: f.writelines(train_data)
    if sec_valid_file:
        with open(sec_valid_file, 'w', encoding='utf-8') as f: f.writelines(valid_data)
    if sec_test_file:
        with open(sec_test_file, 'w', encoding='utf-8') as f: f.writelines(test_data)

split_jsonl(OUTPUT_JSONL, TRAIN_FILE, VALID_FILE, TEST_FILE, PMC_TRAIN_FILE, PMC_VALID_FILE, PMC_TEST_FILE)

Train set: 80 records -> /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/GTC25_DLI/data/training_data/train.jsonl
Validation set: 10 records -> /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/GTC25_DLI/data/training_data/valid.jsonl
Test set: 10 records -> /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/GTC25_DLI/data/training_data/test.jsonl


### Sanitize JSONL Lines
Sanitizes JSONL lines to guarantee valid JSON formatting before feeding to training routines.

In [36]:
import os
import json

def sanitize_jsonl(input_file, output_file):
    """Sanitizes JSONL file safely even when input and output paths are identical."""
    if not os.path.exists(input_file):
        return
    with open(input_file, 'r', encoding='utf-8') as infile:
        lines = infile.readlines()
    
    valid_lines = []
    for line_number, line in enumerate(lines, 1):
        line = line.strip()
        if not line:
            continue
        try:
            json_data = json.loads(line)
            valid_lines.append(json.dumps(json_data, ensure_ascii=False) + '\n')
        except json.JSONDecodeError:
            print(f"Skipping malformed line {line_number} in {input_file}")
            
    with open(output_file, 'w', encoding='utf-8') as outfile:
        outfile.writelines(valid_lines)
    print(f"Sanitized {len(valid_lines)} lines: {output_file}")

# Sanitize MLX files (in-place safe)
sanitize_jsonl(TRAIN_FILE, TRAIN_FILE)
sanitize_jsonl(VALID_FILE, VALID_FILE)
sanitize_jsonl(TEST_FILE, TEST_FILE)

# Sanitize legacy clean files
sanitize_jsonl(PMC_TRAIN_FILE, os.path.join(TRAINING_DATA_DIR, "pmc_train_clean.jsonl"))
sanitize_jsonl(PMC_VALID_FILE, os.path.join(TRAINING_DATA_DIR, "pmc_val_clean.jsonl"))
sanitize_jsonl(PMC_TEST_FILE, os.path.join(TRAINING_DATA_DIR, "pmc_test_clean.jsonl"))

Sanitized 80 lines: /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/GTC25_DLI/data/training_data/train.jsonl
Sanitized 10 lines: /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/GTC25_DLI/data/training_data/valid.jsonl
Sanitized 10 lines: /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/GTC25_DLI/data/training_data/test.jsonl
Sanitized 80 lines: /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/GTC25_DLI/data/training_data/pmc_train_clean.jsonl
Sanitized 10 lines: /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/GTC25_DLI/data/training_data/pmc_val_clean.jsonl
Sanitized 10 lines: /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/GTC25_DLI/data/training_data/pmc_test_clean.jsonl


<br/>
## <font color="#76b900">**2. LoRA Fine-Tuning Execution**</font>

### **Track 1: Local Fine-Tuning on Apple Silicon (MLX - Recommended for Mac)**

Fine-tune `Meta-Llama-3-8B-Instruct-4bit` on Apple Silicon Metal Performance Shaders (MPS) with zero CUDA or Docker overhead:
- **Model**: `mlx-community/Meta-Llama-3-8B-Instruct-4bit` (~4.5GB VRAM)
- **Data**: `../data/training_data/` (`train.jsonl` and `valid.jsonl`)
- **Batch size**: `4`
- **LoRA Layers**: `16`
- **Iterations**: `200` for rapid demo/convergence testing; scale to `1000+` for production.
- **Output Adapter**: `../model/loras/Meta-Llama-3-8B-Instruct-PMC-LoRA-mlx/adapters.npz`

In [40]:
%%bash
# Run Apple Silicon MLX LoRA Fine-Tuning on PMC Clinical Dataset
ARGS=(
  --model mlx-community/Meta-Llama-3-8B-Instruct-4bit
  --data ../data/training_data/
  --train
  --iters 200
  --batch-size 4
  --num-layers 16
  --learning-rate 1e-4
  --mask-prompt
  --grad-checkpoint
  --max-seq-length 2048
  --adapter-path ../model/loras/Meta-Llama-3-8B-Instruct-PMC-LoRA-mlx
  --save-every 50
  --steps-per-eval 50
)

python3 -m mlx_lm lora "${ARGS[@]}"


[transformers] Disabling PyTorch because PyTorch >= 2.5 is required but found 2.2.2


Loading pretrained model


Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 2188.14it/s]


Loading datasets
Training
Trainable parameters: 0.131% (10.486M/8030.261M)
Starting training..., iters: 200


Calculating loss...: 100%|██████████| 2/2 [00:14<00:00,  7.46s/it]


...

Iter 100: Val loss 0.111, Val took 14.332s
Iter 100: Train loss 0.104, Learning Rate 1.000e-04, It/sec 0.044, Tokens/sec 20.438, Trained Tokens 46135, Peak mem 10.429 GB
Iter 100: Saved adapter weights to ../model/loras/Meta-Llama-3-8B-Instruct-PMC-LoRA-mlx/adapters.safetensors and ../model/loras/Meta-Llama-3-8B-Instruct-PMC-LoRA-mlx/0000100_adapters.safetensors.
Iter 110: Train loss 0.082, Learning Rate 1.000e-04, It/sec 0.043, Tokens/sec 19.893, Trained Tokens 50747, Peak mem 10.429 GB
Iter 120: Train loss 0.082, Learning Rate 1.000e-04, It/sec 0.044, Tokens/sec 20.126, Trained Tokens 55362, Peak mem 10.429 GB
Iter 130: Train loss 0.081, Learning Rate 1.000e-04, It/sec 0.044, Tokens/sec 18.859, Trained Tokens 59640, Peak mem 10.429 GB
Iter 140: Train loss 0.076, Learning Rate 1.000e-04, It/sec 0.045, Tokens/sec 22.378, Trained Tokens 64589, Peak mem 10.429 GB


Calculating loss...: 100%|██████████| 2/2 [00:12<00:00,  6.19s/it]


Iter 150: Val loss 0.135, Val took 12.386s
Iter 150: Train loss 0.069, Learning Rate 1.000e-04, It/sec 0.048, Tokens/sec 22.378, Trained Tokens 69271, Peak mem 10.429 GB
Iter 150: Saved adapter weights to ../model/loras/Meta-Llama-3-8B-Instruct-PMC-LoRA-mlx/adapters.safetensors and ../model/loras/Meta-Llama-3-8B-Instruct-PMC-LoRA-mlx/0000150_adapters.safetensors.
Iter 160: Train loss 0.079, Learning Rate 1.000e-04, It/sec 0.047, Tokens/sec 21.523, Trained Tokens 73816, Peak mem 10.429 GB
Iter 170: Train loss 0.061, Learning Rate 1.000e-04, It/sec 0.048, Tokens/sec 21.967, Trained Tokens 78386, Peak mem 10.429 GB
Iter 180: Train loss 0.065, Learning Rate 1.000e-04, It/sec 0.047, Tokens/sec 21.890, Trained Tokens 83043, Peak mem 10.429 GB
Iter 190: Train loss 0.056, Learning Rate 1.000e-04, It/sec 0.048, Tokens/sec 22.054, Trained Tokens 87668, Peak mem 10.429 GB


Calculating loss...: 100%|██████████| 2/2 [00:12<00:00,  6.08s/it]


Iter 200: Val loss 0.129, Val took 12.166s
Iter 200: Train loss 0.059, Learning Rate 1.000e-04, It/sec 0.048, Tokens/sec 22.050, Trained Tokens 92270, Peak mem 10.429 GB
Iter 200: Saved adapter weights to ../model/loras/Meta-Llama-3-8B-Instruct-PMC-LoRA-mlx/adapters.safetensors and ../model/loras/Meta-Llama-3-8B-Instruct-PMC-LoRA-mlx/0000200_adapters.safetensors.
Saved final weights to ../model/loras/Meta-Llama-3-8B-Instruct-PMC-LoRA-mlx/adapters.safetensors.


In [ ]:
import sys
import subprocess

# Alternative: Run MLX LoRA training directly from Python with unified memory optimizations
def run_mlx_lora_python(iters=200, batch_size=4, num_layers=16):
    cmd = [
        sys.executable, "-m", "mlx_lm", "lora",
        "--model", "mlx-community/Meta-Llama-3-8B-Instruct-4bit",
        "--data", TRAINING_DATA_DIR,
        "--train",
        "--iters", str(iters),
        "--batch-size", str(batch_size),
        "--num-layers", str(num_layers),
        "--learning-rate", "1e-4",
        "--mask-prompt",
        "--grad-checkpoint",
        "--max-seq-length", "2048",
        "--adapter-path", LORA_DIR_MLX,
        "--save-every", "50",
        "--steps-per-eval", "50"
    ]
    print(f"Launching MLX LoRA fine-tuning for {iters} iterations on PMC clinical data...")
    result = subprocess.run(cmd, capture_output=False)
    print(f"Training completed with exit code: {result.returncode}")

# Uncomment to execute programmatically from Python:
# run_mlx_lora_python(iters=200)


---
### **Track 2: Scaling to Cloud / Enterprise Multi-GPU Clusters (NVIDIA NeMo)**

When training on multi-node GPU clusters (e.g. 8x H100 / A100), transition to **NVIDIA NeMo Megatron-LM** using Tensor Model Parallelism (TP) and Pipeline Parallelism (PP).

In [ ]:
import os
import urllib.request

# Cluster Step: Download NeMo base model checkpoint
# (Only executed when running in an NVIDIA CUDA / NeMo cluster environment)
nemo_directory = "../model/llama-3-8b-instruct-nemo_v1.0"
nemo_file_path = os.path.join(nemo_directory, "8b_instruct_nemo_bf16.nemo")
nemo_url = "https://api.ngc.nvidia.com/v2/models/org/nvidia/team/nemo/llama-3-8b-instruct-nemo/1.0/files?redirect=true&path=8b_instruct_nemo_bf16.nemo"

RUN_NEMO_PIPELINE = os.getenv("RUN_NEMO_PIPELINE", "0") == "1"

if RUN_NEMO_PIPELINE:
    os.makedirs(nemo_directory, exist_ok=True)
    if not os.path.exists(nemo_file_path):
        print("Downloading NeMo checkpoint from NGC...")
        urllib.request.urlretrieve(nemo_url, nemo_file_path)
    print("NeMo checkpoint ready:", os.listdir(nemo_directory))
else:
    print("Skipping NeMo download (Local execution mode is set to Apple Silicon MLX).")
    print("Set RUN_NEMO_PIPELINE=1 when running on an NVIDIA cluster.")

In [ ]:
%%bash
# Cluster Step: Megatron-LM LoRA fine-tuning via NeMo container
if command -v docker &> /dev/null && docker ps | grep -q "containerB"; then
    docker exec containerB bash -c "
        MODEL='/workspace/model/llama-3-8b-instruct-nemo_v1.0/8b_instruct_nemo_bf16.nemo'
        TRAIN_DS='/workspace/data/training_data/pmc_train_clean.jsonl'
        VALID_DS='/workspace/data/training_data/pmc_val_clean.jsonl'
        OUTPUT_DIR='/workspace/model/Meta-Llama-3-8B-Instruct-PMC-LoRA'
        
        torchrun --nproc_per_node=1 /opt/NeMo/examples/nlp/language_modeling/tuning/megatron_gpt_finetuning.py \
            exp_manager.exp_dir=\${OUTPUT_DIR} \
            trainer.devices=1 \
            trainer.precision=bf16-mixed \
            trainer.max_steps=20 \
            model.restore_from_path=\${MODEL} \
            model.data.train_ds.file_names=[\${TRAIN_DS}] \
            model.data.validation_ds.file_names=[\${VALID_DS}] \
            model.peft.peft_scheme=lora
    "
else
    echo "NVIDIA NeMo container (containerB) is not running on this host. Use MLX above for local Apple Silicon training."
fi

<br/>
## <font color="#76b900">**3. LoRA Model Testing & Clinical Inference**</font>

### **Local Validation on Mac (MLX)**
Once training completes, the adapter weights are saved in `adapters.safetensors`. You can test clinical triplet extraction immediately using:
1. **MLX CLI** (`mlx_lm.generate`)
2. **Python API** (`mlx_lm.load` + `mlx_lm.generate`)


In [41]:
%%bash
# Quick test via MLX CLI on a real clinical patient case
PROMPT="<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Extract all clinical entity-relation-entity triplets from the following patient case as a valid Python list:

This 60-year-old male was hospitalized due to moderate ARDS from COVID-19 with symptoms of fever, dry cough, and dyspnea. Physical therapy was administered.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"

ARGS=(
  --model mlx-community/Meta-Llama-3-8B-Instruct-4bit
  --adapter-path ../model/loras/Meta-Llama-3-8B-Instruct-PMC-LoRA-mlx
  --prompt "${PROMPT}"
  --max-tokens 256
)

python3 -m mlx_lm generate "${ARGS[@]}"


[transformers] Disabling PyTorch because PyTorch >= 2.5 is required but found 2.2.2
Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 1531.79it/s]


[('Patient 87', 'PERSON', 'Has_Age', '60 years', 'METRIC'), ('Patient 87', 'PERSON', 'Has_Gender', 'Male', 'CONCEPT'), ('Patient 87', 'PERSON', 'Diagnosed_With', 'ARDS', 'CONDITION'), ('Patient 87', 'PERSON', 'Presents_With', 'Fever', 'CONDITION'), ('Patient 87', 'PERSON', 'Presents_With', 'Cough', 'CONDITION'), ('Patient 87', 'PERSON', 'Presents_With', 'Dyspnea', 'CONDITION'), ('Patient 87', 'PERSON', 'Receives', 'Physical Therapy', 'TREATMENT')]<|eot_id|><|eot_id|><|start_header_id|>assistant<|end_header_id|>

!!!!!!!!!<|eot_id|><|eot_id|><|start_header_id|>assistant<|end_header_id|>

!!!!!!!! ('Patient 87', 'PERSON', 'Has_Age', '60 years', 'METRIC'), ('Patient 87', 'PERSON', 'Has_Gender', 'Female', 'CONCEPT'), ('Patient 87', 'PERSON', 'Diagnosed_With', 'ARDS', 'CONDITION'), ('Patient 87', 'PERSON', 'Presents_With', '
Prompt: 74 tokens, 42.092 tokens-per-sec
Generation: 256 tokens, 17.105 tokens-per-sec
Peak memory: 5.547 GB


In [44]:
import os
import json
from pprint import pprint

def evaluate_clinical_sample_with_mlx(test_file=TEST_FILE):
    """Evaluate fine-tuned MLX model against a held-out patient case."""
    try:
        from mlx_lm import load, generate
    except ImportError:
        print("mlx_lm is not installed. Install via: pip install mlx-lm")
        return

    adapter_safetensors = os.path.join(LORA_DIR_MLX, "adapters.safetensors")
    adapter_npz = os.path.join(LORA_DIR_MLX, "adapters.npz")
    has_adapter = os.path.exists(adapter_safetensors) or os.path.exists(adapter_npz)
    adapter_arg = LORA_DIR_MLX if has_adapter else None
    
    if adapter_arg:
        print(f"Loading base model + LoRA adapter from: {adapter_arg}")
    else:
        print(f"Notice: LoRA adapter not found in {LORA_DIR_MLX}. Loading base model to test prompt formatting...")

    model, tokenizer = load("mlx-community/Meta-Llama-3-8B-Instruct-4bit", adapter_path=adapter_arg)

    # Read a sample from the test dataset
    if os.path.exists(test_file):
        with open(test_file, 'r', encoding='utf-8') as f:
            sample_line = f.readline()
        sample = json.loads(sample_line)
        raw_text = sample.get("input", "")
        ground_truth = sample.get("output", "")
    else:
        raw_text = "This 60-year-old male was hospitalized due to moderate ARDS from COVID-19 with symptoms of fever, dry cough, and dyspnea."
        ground_truth = "[('Patient 0', 'PERSON', 'Diagnosed_With', 'ARDS', 'CONDITION')]"

    prompt_text = (
        f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
        f"Extract all clinical entity-relation-entity triplets from the following patient case as a valid Python list:\n\n{raw_text}"
        f"<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    )

    print("\n--- GENERATING CLINICAL PREDICTION WITH MLX ---")
    prediction = generate(model, tokenizer, prompt=prompt_text, max_tokens=256, verbose=True)
    
    print("\n" + "="*60)
    print("PREDICTED CLINICAL TRIPLETS:")
    print("="*60)
    print(prediction)
    print("\n" + "="*60)
    print("GROUND TRUTH TARGET TRIPLETS:")
    print("="*60)
    pprint(ground_truth)

# Run local evaluation
# evaluate_clinical_sample_with_mlx()


<br/>
### **Final Validation: Quantitative Benchmark (Base Model without LoRA vs. Fine-Tuned Model with LoRA)**

Evaluate the deterministic extraction quality and schema adherence of the LoRA adapter across held-out patient cases from `test.jsonl`.

This benchmark rigorously evaluates:
1. **Strict Python Parse Rate**: Verifies if the output evaluates directly to a Python `list[tuple]` literal without conversational preamble or markdown fences.
2. **KG 5-Tuple Schema Compliance**: Checks adherence to `(Subject, Subj_Type, Predicate, Object, Obj_Type)` across the standardized clinical ontology (`Has_Age`, `Has_Gender`, `Diagnosed_With`, `Presents_With`, `Receives`).
3. **Semantic Precision, Recall & F1**: Quantifies exact clinical triplet matching against ground truth annotations.
4. **Inference Latency Overhead**: Measures real tokens/second speed on Apple Silicon Metal Performance Shaders.


In [1]:
import os
import sys
import json
import ast
import time
from typing import List, Tuple, Set, Dict, Any

STANDARD_PREDICATES = {"has_age", "has_gender", "diagnosed_with", "presents_with", "receives"}

def _extract_triplets_robust(raw_output: str) -> Tuple[bool, bool, List[Tuple]]:
    """Parse LLM output into triplets and assess strict formatting and schema compliance."""
    text = raw_output.strip()
    if "<|eot_id|>" in text:
        text = text.split("<|eot_id|>")[0].strip()

    is_strict_python = False
    try:
        val = ast.literal_eval(text)
        if isinstance(val, list):
            is_strict_python = True
    except Exception:
        pass

    content_to_parse = text
    if not is_strict_python:
        if "```python" in text:
            content_to_parse = text.split("```python")[1].split("```")[0].strip()
        elif "```" in text:
            content_to_parse = text.split("```")[1].split("```")[0].strip()
        else:
            s = text.find("[")
            e = text.rfind("]")
            if s != -1 and e != -1 and e > s:
                content_to_parse = text[s:e+1]

    parsed_list = []
    try:
        val = ast.literal_eval(content_to_parse)
        if isinstance(val, list):
            parsed_list = val
    except Exception:
        pass

    valid_tuples = []
    schema_compliant_count = 0
    for item in parsed_list:
        if isinstance(item, (list, tuple)) and len(item) == 5:
            subj, stype, pred, obj, otype = [str(x).strip() for x in item]
            valid_tuples.append((subj, stype, pred, obj, otype))
            if pred.lower() in STANDARD_PREDICATES:
                schema_compliant_count += 1
        elif isinstance(item, (list, tuple)) and len(item) == 3:
            s, p, o = [str(x).strip() for x in item]
            valid_tuples.append((s, "UNKNOWN", p, o, "UNKNOWN"))

    is_schema_compliant = (len(valid_tuples) > 0 and schema_compliant_count == len(valid_tuples))
    return is_strict_python, is_schema_compliant, valid_tuples

def _normalize_semantic_triplets(triplets: List[Tuple]) -> Set[Tuple]:
    """Normalize subject 'Patient <id>' -> 'patient' to isolate clinical entity/relation accuracy."""
    res = set()
    for item in triplets:
        if len(item) >= 5:
            subj = "patient"
            stype = str(item[1]).strip().lower()
            pred = str(item[2]).strip().lower()
            obj = str(item[3]).strip().lower()
            otype = str(item[4]).strip().lower()
            res.add((subj, stype, pred, obj, otype))
    return res

def _compute_prf1(pred_set: Set[Tuple], gt_set: Set[Tuple]) -> Dict[str, float]:
    if not pred_set and not gt_set:
        return {"precision": 1.0, "recall": 1.0, "f1": 1.0}
    if not pred_set:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    if not gt_set:
        return {"precision": 0.0, "recall": 1.0, "f1": 0.0}

    tp = len(pred_set.intersection(gt_set))
    fp = len(pred_set - gt_set)
    fn = len(gt_set - pred_set)

    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
    return {"precision": p, "recall": r, "f1": f1}

def compare_base_vs_lora_models(test_file: str = TEST_FILE, num_samples: int = 5, model_id: str = "mlx-community/Meta-Llama-3-8B-Instruct-4bit"):
    """Run quantitative side-by-side benchmark between Base Model and LoRA Fine-Tuned Model."""
    try:
        from mlx_lm import load, generate
    except ImportError:
        print("mlx_lm is not installed. Run: pip install mlx-lm")
        return

    if not os.path.exists(test_file):
        print(f"Test dataset not found at {test_file}.")
        return

    with open(test_file, "r", encoding="utf-8") as f:
        samples = [json.loads(line) for line in f if line.strip()][:num_samples]

    adapter_path = LORA_DIR_MLX
    if not os.path.exists(adapter_path):
        print(f"LoRA adapter not found at {adapter_path}. Train the adapter first in Step 2.")
        return

    print(f"Starting Quantitative Benchmark on {len(samples)} held-out clinical cases...")

    # Phase 1: Base Model (No LoRA)
    print("\n" + "="*70)
    print("PHASE 1: RUNNING BASE MODEL (WITHOUT LORA)")
    print("="*70)
    base_model, base_tok = load(model_id)
    
    base_results = []
    for i, s in enumerate(samples):
        prompt = (
            f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
            f"Extract all clinical entity-relation-entity triplets from the following patient case as a valid Python list:\n\n{s['input']}"
            f"<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
        )
        t0 = time.time()
        output = generate(base_model, base_tok, prompt=prompt, max_tokens=256, verbose=False)
        dt = time.time() - t0
        
        is_strict, is_schema, preds = _extract_triplets_robust(output)
        _, _, gts = _extract_triplets_robust(s.get("output", ""))
        metrics = _compute_prf1(_normalize_semantic_triplets(preds), _normalize_semantic_triplets(gts))
        
        base_results.append({"strict": is_strict, "schema": is_schema, "metrics": metrics, "time": dt, "output": output})
        print(f"  Sample {i+1:02d} | Strict Python: {str(is_strict):5s} | Schema: {str(is_schema):5s} | P: {metrics['precision']:.2f} | R: {metrics['recall']:.2f} | F1: {metrics['f1']:.2f} ({dt:.2f}s)")

    del base_model
    del base_tok
    import gc
    gc.collect()

    # Phase 2: Fine-Tuned Model (With LoRA)
    print("\n" + "="*70)
    print("PHASE 2: RUNNING FINE-TUNED MODEL (WITH LORA)")
    print("="*70)
    lora_model, lora_tok = load(model_id, adapter_path=adapter_path)
    
    lora_results = []
    for i, s in enumerate(samples):
        prompt = (
            f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
            f"Extract all clinical entity-relation-entity triplets from the following patient case as a valid Python list:\n\n{s['input']}"
            f"<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
        )
        t0 = time.time()
        output = generate(lora_model, lora_tok, prompt=prompt, max_tokens=256, verbose=False)
        dt = time.time() - t0
        
        is_strict, is_schema, preds = _extract_triplets_robust(output)
        _, _, gts = _extract_triplets_robust(s.get("output", ""))
        metrics = _compute_prf1(_normalize_semantic_triplets(preds), _normalize_semantic_triplets(gts))
        
        lora_results.append({"strict": is_strict, "schema": is_schema, "metrics": metrics, "time": dt, "output": output})
        print(f"  Sample {i+1:02d} | Strict Python: {str(is_strict):5s} | Schema: {str(is_schema):5s} | P: {metrics['precision']:.2f} | R: {metrics['recall']:.2f} | F1: {metrics['f1']:.2f} ({dt:.2f}s)")

    del lora_model
    del lora_tok
    gc.collect()

    # Aggregate Metrics
    N = len(samples)
    b_strict = sum(1 for r in base_results if r["strict"]) / N * 100
    l_strict = sum(1 for r in lora_results if r["strict"]) / N * 100
    b_schema = sum(1 for r in base_results if r["schema"]) / N * 100
    l_schema = sum(1 for r in lora_results if r["schema"]) / N * 100
    b_p = sum(r["metrics"]["precision"] for r in base_results) / N
    l_p = sum(r["metrics"]["precision"] for r in lora_results) / N
    b_r = sum(r["metrics"]["recall"] for r in base_results) / N
    l_r = sum(r["metrics"]["recall"] for r in lora_results) / N
    b_f1 = sum(r["metrics"]["f1"] for r in base_results) / N
    l_f1 = sum(r["metrics"]["f1"] for r in lora_results) / N
    b_time = sum(r["time"] for r in base_results) / N
    l_time = sum(r["time"] for r in lora_results) / N

    print("\n" + "="*75)
    print("FINAL QUANTITATIVE BENCHMARK: BASE MODEL vs. LORA FINE-TUNED")
    print("="*75)
    print(f"| Metric                              | Without LoRA (Base) | With LoRA (Fine-Tuned) | Absolute Delta |")
    print(f"|-------------------------------------|---------------------|------------------------|----------------|")
    print(f"| Strict Python List Parse Rate       | {b_strict:18.1f}% | {l_strict:21.1f}% | {l_strict - b_strict:+13.1f}% |")
    print(f"| KG Schema Compliance (5-tuple)      | {b_schema:18.1f}% | {l_schema:21.1f}% | {l_schema - b_schema:+13.1f}% |")
    print(f"| Triplet Semantic Precision          | {b_p:19.3f} | {l_p:22.3f} | {l_p - b_p:+14.3f} |")
    print(f"| Triplet Semantic Recall             | {b_r:19.3f} | {l_r:22.3f} | {l_r - b_r:+14.3f} |")
    print(f"| Triplet Semantic F1 Score           | {b_f1:19.3f} | {l_f1:22.3f} | {l_f1 - b_f1:+14.3f} |")
    print(f"| Average Inference Latency           | {b_time:17.2f}s | {l_time:20.2f}s | {l_time - b_time:+13.2f}s |")
    print("="*75)

# Execute the benchmark comparison across held-out test cases
compare_base_vs_lora_models(num_samples=5)


[transformers] Disabling PyTorch because PyTorch >= 2.5 is required but found 2.2.2
Reading test data from: /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/GTC25_DLI/data/training_data/test.jsonl
Total test samples to evaluate: 10

RUNNING INFERENCE: BASE MODEL (WITHOUT LORA)

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 2332.98it/s]
Loaded mlx-community/Meta-Llama-3-8B-Instruct-4bit in 4.17s
  Sample 01 | Strict Python: False | Schema: False | P: 0.00 | R: 0.00 | F1: 0.00 (15.29s)
  Sample 02 | Strict Python: False | Schema: False | P: 0.00 | R: 0.00 | F1: 0.00 (13.68s)
  Sample 03 | Strict Python: False | Schema: False | P: 0.00 | R: 0.00 | F1: 0.00 (14.36s)
  Sample 04 | Strict Python: False | Schema: False | P: 0.00 | R: 0.00 | F1: 0.00 (14.17s)
  Sample 05 | Strict Python: False | Schema: False | P: 0.00 | R: 0.00 | F1: 0.00 (14.75s)
  Sample 06 | Strict Python: False | Schema: False | P: 0.00 | R: 0.00 | F1: 0.00 (15.02s)
  Sample 07 | Strict Pyt

<br/>
## <font color="#76b900">**4. Deployment & Serving Options**</font>

### **Local Deployment: Fusing the Adapter**
Fuse LoRA weights directly into the base model to produce a standalone model for zero-overhead inference or Ollama/GGUF export:
```bash
mlx_lm.fuse \
  --model mlx-community/Meta-Llama-3-8B-Instruct-4bit \
  --adapter-path ../model/loras/Meta-Llama-3-8B-Instruct-PMC-LoRA-mlx \
  --save-path ../model/Meta-Llama-3-8B-Instruct-PMC-Fused
```

---

### **Enterprise Deployment: NVIDIA NIM Microservice**
When deployed on production clusters, serve the model and LoRA adapter via **NVIDIA NIM** (port 8000).

In [43]:
import requests
from pprint import pprint

# Querying NIM or a local OpenAI-compatible server (e.g., vLLM / Ollama / NIM)
def query_model_endpoint(endpoint_url="http://localhost:8000/v1/chat/completions", patient_text="This 60-year-old male was hospitalized due to moderate ARDS from COVID-19."):
    headers = {
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": "Meta-Llama-3-8B-Instruct-PMC-LoRA",
        "messages": [
            {
                "role": "user",
                "content": f"Extract all clinical entity-relation-entity triplets from the following patient case as a valid Python list:\n\n{patient_text}"
            }
        ],
        "max_tokens": 256,
        "temperature": 0.1
    }
    
    try:
        response = requests.post(endpoint_url, headers=headers, json=payload, timeout=5)
        if response.status_code == 200:
            result = response.json()
            pprint(result["choices"][0]["message"]["content"])
        else:
            print(f"Server responded with code {response.status_code}: {response.text}")
    except requests.exceptions.ConnectionError:
        print(f"Notice: No active inference server detected at {endpoint_url}.")
        print("To run local inference on Mac, use the MLX evaluation function above.")

# Test endpoint connectivity (non-blocking)
query_model_endpoint()

Notice: No active inference server detected at http://localhost:8000/v1/chat/completions.
To run local inference on Mac, use the MLX evaluation function above.
